In [1]:
import urllib
from transformers import pipeline
import os
import json
from datetime import datetime
from dataclasses import dataclass
import gradio as gr
import logging
from sentence_transformers import SentenceTransformer, util
from gliner import GLiNER
import xml.etree.ElementTree as ET
from dotenv import load_dotenv

load_dotenv()

/home/rakesh/Downloads/ShortTok/1.VoiceModel/VoiceSearch/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/rakesh/Downloads/ShortTok/1.VoiceModel/VoiceSearch/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


True

In [2]:
@dataclass
class VoiceSearch:
    user_query_audio: str | None = None
    user_query_text: str | None = None
    intent: str | None = None
    searchable_query: str | None = None
    search_results: list | None = None
    latency_transcription: str | None = None
    latency_intent_detection: str | None = None
    latency_query_extraction: str | None = None
    latency_search: str | None = None
    latency_overall: str | None = None

    def __str__(self):
        return json.dumps(self.__dict__, indent=2)

def jsonify(voice_search):
    return str(voice_search)

In [3]:
os.makedirs("logs", exist_ok=True)
logging.basicConfig(filename="logs/app.log", level=logging.INFO, format="%(message)s")
voice_search_logger = logging.getLogger("voice_search")
voice_search_logger.propagate = False  # prevents log messages from being passed to the root logger (avoids duplicate logs)
voice_search_logger.setLevel(logging.INFO)
if not voice_search_logger.handlers:
    handler = logging.FileHandler("logs/voice_search.log")
    handler.setFormatter(logging.Formatter("%(message)s"))
    voice_search_logger.addHandler(handler)

In [4]:
asr = pipeline(task="automatic-speech-recognition",
               model="distil-whisper/distil-small.en",
               model_kwargs={"cache_dir": "models"})

print('asr_freq:', asr.feature_extractor.sampling_rate)

Loading weights: 100%|██████████| 287/287 [00:00<00:00, 15011.98it/s]


asr_freq: 16000


In [5]:
intent_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2",
                                    device="cuda",
                                    model_kwargs={"cache_dir": "models"})

intents = ["news", "ecommerce"]
intent_descriptions = ["latest news or headlines about a person or topic",
                       "shop for a product on Amazon, eBay, or Flipkart"]
intent_emb = intent_model.encode(intent_descriptions, normalize_embeddings=True)

Batches: 100%|██████████| 1/1 [00:00<00:00,  4.46it/s]


In [6]:
ner_model = GLiNER.from_pretrained("urchade/gliner_small-v2.1",
                                    map_location="cuda",
                                    cache_dir="models")

/home/rakesh/Downloads/ShortTok/1.VoiceModel/VoiceSearch/.venv/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:190: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 1227.30it/s]


In [7]:
def save_audio(filepath, text):
    if filepath is not None:
        audio_dir = os.path.abspath("logs/audio")
        src = os.path.abspath(filepath)
        if src.startswith(audio_dir + os.sep):
            stem = "".join(c for c in text.strip() if c.isalnum() or c in " ,.'-")[:80].strip() or "audio"
            ext = os.path.splitext(src)[1]
            dest = os.path.join(audio_dir, stem + ext)
            i = 2
            while os.path.exists(dest):
                dest = os.path.join(audio_dir, f"{stem} {i}{ext}")
                i += 1
            os.replace(src, dest)
            os.rmdir(os.path.dirname(src))
            filepath = dest
    return None if filepath is None else os.path.relpath(filepath)
    
def transcribe_speech(voice_search):
    filepath = voice_search.user_query_audio
    latency_ms = None
    if filepath is None:
        gr.Warning("No audio found, please retry.")
        voice_search.user_query_text = ""
    else:
        started = datetime.now()
        output = asr(filepath)
        latency_ms = round((datetime.now() - started).total_seconds() * 1000)
        voice_search.user_query_text = output["text"]
        voice_search.latency_transcription = f"{latency_ms} ms"
    voice_search.user_query_audio = save_audio(filepath, voice_search.user_query_text)
    return jsonify(voice_search)

In [8]:
def detect_intent(voice_search):
    started = datetime.now()
    emb = intent_model.encode(voice_search.user_query_text, normalize_embeddings=True)
    scores = util.cos_sim(emb, intent_emb)[0]
    probs = scores.softmax(dim=0)
    i = int(probs.argmax())
    voice_search.intent = intents[i]
    voice_search.latency_intent_detection = f"{round((datetime.now() - started).total_seconds() * 1000)} ms"
    return jsonify(voice_search)

In [9]:
INTENT_LABELS = {
    "news": ["person", "organization", "place"],
    "ecommerce": ["item", "product"],
}

def extract_query(voice_search):
    started = datetime.now()
    text = voice_search.user_query_text
    intent = voice_search.intent
    spans = list(dict.fromkeys(
        e["text"] for e in ner_model.predict_entities(text, INTENT_LABELS[intent], threshold=0.4)
    ))
    if spans:
        emb = intent_model.encode([text, *spans], normalize_embeddings=True)
        target = intent_emb[intents.index(intent)]
        scores = util.cos_sim(emb[0], emb[1:])[0] + 2 * util.cos_sim(target, emb[1:])[0]
        probs = scores.softmax(dim=0)
        i = int(probs.argmax())
        voice_search.searchable_query = spans[i]
    else:
        voice_search.searchable_query = ""
    voice_search.latency_query_extraction = f"{round((datetime.now() - started).total_seconds() * 1000)} ms"
    return jsonify(voice_search)

In [10]:
cases = [
    ("Well tell me some news about Tom Cruise.", "news", "Tom Cruise"),
    ("What's up with Trump these days?", "news", "Trump"),
    ("Show me some t-shirts from Amazon.com", "ecommerce", "t-shirts"),
    ("get me some nyke shoes from amazon.com", "ecommerce", "nyke shoes"),
    ("find me some shoes similar to elon musk", "ecommerce", "shoes"),
    ("Tell me the latest headlines about Barack Obama.", "news", "Barack Obama"),
    ("What is happening in New York these days?", "news", "New York"),
    ("Any recent news about NASA?", "news", "NASA"),
    ("Find me wireless headphones from eBay.", "ecommerce", "wireless headphones"),
    ("Show me Adidas running shoes.", "ecommerce", "Adidas running shoes"),
    ("I want a jacket like the one Taylor Swift wore.", "ecommerce", "jacket"),
    ("What's the news about Apple's new iPhone?", "news", "Apple"),
    ("get me some addidas sneakers from flipkart", "ecommerce", "addidas sneakers"),
    ("Trump news", "news", "Trump"),
    ("looking for a cheap office chair", "ecommerce", "office chair"),
]

ok = 0
for text, actual_intent, actual_query in cases:
    record = VoiceSearch(user_query_text=text)
    detect_intent(record)
    extract_query(record)
    intent = record.intent
    query = record.searchable_query
    match = intent == actual_intent and query == actual_query
    ok += match
    print(text)
    print(f"{'ok' if match else 'WRONG'} ----- intent: {intent}, query: {query!r}")
    print()
print(f"ok: {ok}/{len(cases)}")

Batches: 100%|██████████| 1/1 [00:00<00:00, 285.58it/s]


Well tell me some news about Tom Cruise.
ok ----- intent: news, query: 'Tom Cruise'



Batches: 100%|██████████| 1/1 [00:00<00:00, 264.16it/s]


What's up with Trump these days?
ok ----- intent: news, query: 'Trump'



Batches: 100%|██████████| 1/1 [00:00<00:00, 274.89it/s]


...

get me some addidas sneakers from flipkart
ok ----- intent: ecommerce, query: 'addidas sneakers'



Batches: 100%|██████████| 1/1 [00:00<00:00, 385.93it/s]


Trump news
ok ----- intent: news, query: 'Trump'



Batches: 100%|██████████| 1/1 [00:00<00:00, 265.33it/s]

looking for a cheap office chair
ok ----- intent: ecommerce, query: 'office chair'

ok: 15/15


In [11]:
PDP_PATH = {
    "amazon.com": "inurl:/dp/",
    "ebay.com": "inurl:/itm/",
    "walmart.com": "inurl:/ip/",
    "target.com": "inurl:/p/",
    "bestbuy.com": "inurl:.p",
    "etsy.com": "inurl:/listing/",
    "flipkart.com": "inurl:/p/",
    "myntra.com": "inurl:/buy",
    "nike.com": "inurl:/t/",
}

def store_site(text):
    orgs = ner_model.predict_entities(text, ["organization"], threshold=0.4)
    people = {e["text"].lower() for e in ner_model.predict_entities(text, ["person"], threshold=0.3)}
    for org in orgs:
        if org["text"].lower() in people:
            continue
        name = org["text"].lower().removeprefix("www.").rstrip(".")
        return name if "." in name else f"{name}.com"
    return None

def product_query(text, searchable_query):
    site = store_site(text)
    if not site:
        return f"buy {searchable_query}"
    return f"buy {searchable_query} site:{site} {PDP_PATH.get(site, '')}".rstrip()

def news_hits(searchable_query):
    url = "https://news.google.com/rss/search?" + urllib.parse.urlencode({"q": searchable_query, "hl": "en-US", "gl": "US", "ceid": "US:en"})
    with urllib.request.urlopen(url, timeout=5) as response:
        root = ET.fromstring(response.read())
    return [{"title": item.findtext("title"), "url": item.findtext("link")} for item in root.findall("./channel/item")[:5]]

def product_hits(text, searchable_query):
    url = "https://api.search.brave.com/res/v1/web/search?" + urllib.parse.urlencode({"q": product_query(text, searchable_query), "count": 5})
    request = urllib.request.Request(url, headers={"Accept": "application/json", "X-Subscription-Token": os.environ.get("BRAVE_API_KEY")})
    with urllib.request.urlopen(request, timeout=5) as response:
        data = json.load(response)
    return [{"title": hit["title"], "url": hit["url"], "description": hit.get("description")} for hit in data.get("web", {}).get("results", [])[:5]]

def search_results(voice_search):
    started = datetime.now()
    if voice_search.intent == "news":
        hits = news_hits(voice_search.searchable_query)
    else:
        hits = product_hits(voice_search.user_query_text, voice_search.searchable_query)
    voice_search.search_results = hits
    voice_search.latency_search = f"{round((datetime.now() - started).total_seconds() * 1000)} ms"
    parts = [
        voice_search.latency_transcription,
        voice_search.latency_intent_detection,
        voice_search.latency_query_extraction,
        voice_search.latency_search,
    ]
    voice_search.latency_overall = f"{sum(int(part.split()[0]) for part in parts if part)} ms"
    return jsonify(voice_search)

In [12]:
voice_search = VoiceSearch(user_query_text="Well tell me some news about Tom Cruise.")
print(voice_search)
detect_intent(voice_search)
print(voice_search)
extract_query(voice_search)
print(voice_search)
search_results(voice_search)
print(voice_search)

{
  "user_query_audio": null,
  "user_query_text": "Well tell me some news about Tom Cruise.",
  "intent": null,
  "searchable_query": null,
  "search_results": null,
  "latency_transcription": null,
  "latency_intent_detection": null,
  "latency_query_extraction": null,
  "latency_search": null,
  "latency_overall": null
}


Batches: 100%|██████████| 1/1 [00:00<00:00, 371.28it/s]


{
  "user_query_audio": null,
  "user_query_text": "Well tell me some news about Tom Cruise.",
  "intent": "news",
  "searchable_query": null,
  "search_results": null,
  "latency_transcription": null,
  "latency_intent_detection": "6 ms",
  "latency_query_extraction": null,
  "latency_search": null,
  "latency_overall": null
}


Batches: 100%|██████████| 1/1 [00:00<00:00, 347.21it/s]


{
  "user_query_audio": null,
  "user_query_text": "Well tell me some news about Tom Cruise.",
  "intent": "news",
  "searchable_query": "Tom Cruise",
  "search_results": null,
  "latency_transcription": null,
  "latency_intent_detection": "6 ms",
  "latency_query_extraction": "13 ms",
  "latency_search": null,
  "latency_overall": null
}
{
  "user_query_audio": null,
  "user_query_text": "Well tell me some news about Tom Cruise.",
  "intent": "news",
  "searchable_query": "Tom Cruise",
  "search_results": [
    {
      "title": "'Digger': Tom Cruise Movie Bombs at Box Office, Will Lose $150 Million - deadline.com",
      "url": "https://news.google.com/rss/articles/CBMiekFVX3lxTFBiUlBBbDhReUowdkJsS0JaeW9lN1RxX2dVbkk1VUdSWjI2VU1DUzUzU0dLVmZBWVB0ZUNBR1hYVEx1ekU1OHo3dEJ0NV9xdy04ekktbzlxSjZRSEJ0SldFdXB0MkE5Z3NlNGlBMFlXSC16Wk4tSW5sbUJB?oc=5"
    },
    {
      "title": "Tom Cruise's Best Non-Action Movies, Definitively Ranked - GQ",
      "url": "https://news.google.com/rss/articles/CBMiak

In [13]:
voice_search = VoiceSearch(user_query_text="Show me some t-shirts from Amazon.com")
print(voice_search)
detect_intent(voice_search)
print(voice_search)
extract_query(voice_search)
print(voice_search)
search_results(voice_search)
print(voice_search)

{
  "user_query_audio": null,
  "user_query_text": "Show me some t-shirts from Amazon.com",
  "intent": null,
  "searchable_query": null,
  "search_results": null,
  "latency_transcription": null,
  "latency_intent_detection": null,
  "latency_query_extraction": null,
  "latency_search": null,
  "latency_overall": null
}


Batches: 100%|██████████| 1/1 [00:00<00:00, 251.61it/s]


{
  "user_query_audio": null,
  "user_query_text": "Show me some t-shirts from Amazon.com",
  "intent": "ecommerce",
  "searchable_query": null,
  "search_results": null,
  "latency_transcription": null,
  "latency_intent_detection": "8 ms",
  "latency_query_extraction": null,
  "latency_search": null,
  "latency_overall": null
}


Batches: 100%|██████████| 1/1 [00:00<00:00, 273.64it/s]

{
  "user_query_audio": null,
  "user_query_text": "Show me some t-shirts from Amazon.com",
  "intent": "ecommerce",
  "searchable_query": "t-shirts",
  "search_results": null,
  "latency_transcription": null,
  "latency_intent_detection": "8 ms",
  "latency_query_extraction": "15 ms",
  "latency_search": null,
  "latency_overall": null
}


{
  "user_query_audio": null,
  "user_query_text": "Show me some t-shirts from Amazon.com",
  "intent": "ecommerce",
  "searchable_query": "t-shirts",
  "search_results": [
    {
      "title": "Amazon.com: T-shirt, latest T-shirts, new brand, clothes : Clothing, Shoes & Jewelry",
      "url": "https://www.amazon.com/T-shirt-latest-T-shirts-brand-clothes/dp/B07JP42H9X",
      "description": "<strong>Buy</strong> <strong>T</strong>-<strong>shirt</strong>, latest <strong>T</strong>-<strong>shirts</strong>, new brand, clothes: Shop top fashion brands Graphic Tees at Amazon.com \u2713 FREE DELIVERY and Returns possible on eligible purchases"
    },
    {
      "title": "Amazon.com: Custom T-Shirt with Your Image for Men, Women and Kids by Modify by Amazon Merch on Demand : Clothing, Shoes & Jewelry",
      "url": "https://www.amazon.com/Custom-T-Shirt-Your-Image-Green/dp/B09W4DK44V",
      "description": "Amazon.com: Custom <strong>T</strong>-<strong>Shirt</strong> with Your Image for Men,

In [14]:
def inference_hardware():
    on_gpu = asr.device.type == "cuda"
    in_colab = "COLAB_RELEASE_TAG" in os.environ
    if on_gpu and in_colab:
        return "colab gpu"
    if on_gpu:
        return "local gpu"
    return "cpu"

In [ ]:
def run_voice_search(filepath):
    voice_search = VoiceSearch(user_query_audio=filepath)
    transcribe_speech(voice_search)
    if voice_search.user_query_text:
        detect_intent(voice_search)
        extract_query(voice_search)
        try:
            search_results(voice_search)
        except Exception as error:
            voice_search.search_results = str(error)
            parts = [voice_search.latency_transcription, voice_search.latency_intent_detection, voice_search.latency_query_extraction]
            voice_search.latency_overall = f"{sum(int(part.split()[0]) for part in parts if part)} ms"

    voice_search_logger.info(json.dumps({"time": datetime.now().isoformat(), **voice_search.__dict__,
                                         "model": asr.model.name_or_path, "hardware": inference_hardware()}))

    def component(value, name, latency):
        return gr.update(value=value, label=name if not latency else f"{name} ({latency})")
        
    return (
        component(voice_search.user_query_text, "Transcription", voice_search.latency_transcription),
        component(voice_search.intent, "Intent", voice_search.latency_intent_detection),
        component(voice_search.searchable_query, "Searchable query", voice_search.latency_query_extraction),
        component(voice_search.search_results, "Search results", voice_search.latency_search),
        voice_search.latency_overall,
    )

def voice_search_ui(sources):
    return gr.Interface(
        fn=run_voice_search,
        inputs=gr.Audio(sources=sources, type="filepath"),
        outputs=[
            gr.Textbox(label="Transcription", lines=3),
            gr.Textbox(label="Intent"),
            gr.Textbox(label="Searchable query"),
            gr.JSON(label="Search results"),
            gr.Textbox(label="Overall latency"),
        ],
    )

mic_transcribe = voice_search_ui("microphone")
os.makedirs("logs/audio", exist_ok=True)
mic_transcribe.input_components[0].GRADIO_CACHE = os.path.abspath("logs/audio")

file_transcribe = voice_search_ui("upload")

demo = gr.Blocks()

with demo:
    gr.TabbedInterface(
        [mic_transcribe,
         file_transcribe],
        ["Transcribe Microphone",
         "Transcribe Audio File"],
    )

demo.launch(share=True,
            server_port=int(os.environ.get('PORT1', 7680)))

* Running on local URL:  http://127.0.0.1:7680
* Running on public URL: https://2b331ba172948b3367.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
Batches: 100%|██████████| 1/1 [00:00<00:00, 245.31it/s]


In [16]:
demo.close()

Closing server running on port: 7680
